# 05 — Search improvement and evaluation

Run top to bottom in **Google Colab** (T4 recommended). Mount the Drive containing notebooks 03–04's existing artifacts; only query encoders are loaded. No product embeddings, BM25 objects, or FAISS indexes are rebuilt. Notebooks 01–04 remain unchanged.

The previous run reported Precision@10 **0.5875 → 0.5875**, MRR@10 **0.6458 → 0.6146**, and nDCG@10 **0.5791 → 0.6043**. These are historical results, not results of this revision. Old outputs have been cleared. Run this notebook to measure the revised systems; no improvement is assumed.

The old rules matched incidental title mentions (speakers mentioning power banks, organizers mentioning cables, mouse feet mentioning gaming models). They also rewarded generic words and used model-specific aliases. This revision uses catalog-derived category phrases, primary-title evidence, weighted attribute coverage, and a small development-only parameter comparison. No brands, model IDs, or individual query exceptions enter ranking.


In [1]:
%pip -q install "transformers>=4.44,<5" "faiss-cpu>=1.8" "rank-bm25>=0.2" pandas pyarrow numpy torch pillow


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import json, re, math, pickle, hashlib, unicodedata, platform
from pathlib import Path
from collections import Counter, defaultdict
from functools import lru_cache
from importlib.metadata import version

import numpy as np
import pandas as pd
import faiss
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from IPython.display import display
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ElectroLens')
OUT = ROOT / 'search_assets'
SAVE = ROOT / 'search_improvement'
E5_ID = 'intfloat/multilingual-e5-small'
CLIP_ID = 'openai/clip-vit-base-patch32'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
K, CANDIDATE_K, RRF_K = 10, 100, 60
print('Device:', DEVICE, '| Read existing assets from:', OUT)


Mounted at /content/drive
Device: cuda | Read existing assets from: /content/drive/MyDrive/ElectroLens/search_assets


## 1 · Load and verify existing artifacts

Paths, text construction, model IDs, and row conventions follow notebooks 01–04. E5 FAISS positions are catalog rows; CLIP positions must be translated through `clip_image_row_ids.npy`. Load only your own trusted BM25 pickle. Validation fails on missing or mismatched assets rather than creating replacements.


In [3]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def bm25_tokens(text):
    # EXACT tokenizer used to build notebook 04's saved BM25.
    return re.findall(r'\w+', str(text).lower(), flags=re.UNICODE)

def clean_field(value, limit=350):
    if value is None or isinstance(value, (list, dict)):
        return ''
    text = re.sub(r'\s+', ' ', str(value)).strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text[:limit]

def indexed_text(row):
    fields = [clean_field(row['title'], 180)]
    for name in ('store', 'product_group', 'description'):
        if name in row:
            value = clean_field(row[name], 250)
            if value:
                fields.append(value)
    return ' | '.join(fields)

required = ['products_search.parquet', 'manifest.json', 'bm25.pkl',
            'e5_embeddings.npy', 'e5_text.faiss', 'clip_image_embeddings.npy',
            'clip_image_valid.npy', 'clip_image.faiss', 'clip_image_row_ids.npy']
missing = [name for name in required if not (OUT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Existing notebook 03–04 artifacts missing: {missing}')

products = pd.read_parquet(OUT / 'products_search.parquet').reset_index(drop=True)
require({'parent_asin', 'title', 'subcategory', 'cat_path', 'product_group', 'image_url'}
        <= set(products.columns), 'Catalog schema differs from notebooks 01–04')
require(len(products) > 0 and products.parent_asin.notna().all()
        and products.parent_asin.is_unique, 'Invalid product IDs')
products['parent_asin'] = products.parent_asin.astype(str)
asins = products.parent_asin.to_numpy()
catalog_hash = hashlib.sha256('\n'.join(asins).encode()).hexdigest()[:16]
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
require(manifest['catalog_hash'] == catalog_hash and manifest['n_products'] == len(products),
        'Catalog ID/order mismatch')
require(manifest['e5_model'] == E5_ID and manifest['clip_model'] == CLIP_ID,
        'Query models must match saved embeddings')
e5_vectors = np.load(OUT / 'e5_embeddings.npy', mmap_mode='r', allow_pickle=False)
image_vectors = np.load(OUT / 'clip_image_embeddings.npy', mmap_mode='r', allow_pickle=False)
image_valid = np.load(OUT / 'clip_image_valid.npy', allow_pickle=False)
image_rows = np.load(OUT / 'clip_image_row_ids.npy', allow_pickle=False)
e5_index = faiss.read_index(str(OUT / 'e5_text.faiss'))
image_index = faiss.read_index(str(OUT / 'clip_image.faiss'))
require(e5_vectors.shape == (len(products), 384)
        and image_vectors.shape == (len(products), 512), 'Embedding dimensions differ')
require(image_valid.shape == (len(products),) and image_valid.dtype == np.bool_,
        'Invalid image validity mask')
require(image_rows.ndim == 1 and np.issubdtype(image_rows.dtype, np.integer)
        and np.array_equal(image_rows, np.flatnonzero(image_valid)), 'Image row mapping differs')
require(e5_index.ntotal == len(products) and e5_index.d == 384
        and image_index.ntotal == len(image_rows) and image_index.d == 512
        and image_index.ntotal > 0, 'FAISS dimensions/counts differ')
require(e5_index.metric_type == faiss.METRIC_INNER_PRODUCT
        and image_index.metric_type == faiss.METRIC_INNER_PRODUCT, 'Expected inner-product indexes')

# Sample reconstruction validates vector-to-row correspondence without re-adding vectors.
for idx, vectors, rows in ((e5_index, e5_vectors, np.arange(len(products))),
                           (image_index, image_vectors, image_rows)):
    for pos in np.linspace(0, idx.ntotal - 1, min(16, idx.ntotal), dtype=int):
        vector = np.asarray(vectors[int(rows[pos])], dtype='float32')
        require(np.isfinite(vector).all() and np.isclose(np.linalg.norm(vector), 1, atol=1e-3),
                'Invalid normalized embedding')
        require(np.allclose(idx.reconstruct(int(pos)), vector, atol=1e-6),
                'FAISS vector/row mismatch')

with (OUT / 'bm25.pkl').open('rb') as handle:
    bm25 = pickle.load(handle)
require(bm25.corpus_size == len(products), 'BM25 catalog count differs')
for i, row in products.iterrows():
    require(dict(Counter(bm25_tokens(indexed_text(row)))) == bm25.doc_freqs[i],
            f'BM25 tokenizer/text/row mismatch at row {i}')
print('Validated catalog:', len(products), '| valid images:', len(image_rows), '| hash:', catalog_hash)


Validated catalog: 13328 | valid images: 13328 | hash: 4f1e28ecb05ad3eb


## 2 · Query encoders and metadata features

E5 retains `query: `, masked mean pooling and 256-token truncation. CLIP loads lazily for image tests. Only **new queries** are encoded. Canonicalization is confined to reranking: Unicode dashes, USB-C/Type-C spellings, simple plural forms. Saved BM25 always receives its original tokenizer.

Category suffix phrases are derived from the existing catalog, not evaluation queries. Primary title ends before compatibility/use clauses; full-title coverage has less weight. IDF weights come from catalog titles. These heuristics can miss synonyms, noisy categories, or non-English phrasing, so the original RRF remains a selectable control.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(E5_ID)
e5_model = AutoModel.from_pretrained(E5_ID).to(DEVICE).eval()
require(e5_model.config.hidden_size == e5_index.d, 'E5 query dimension mismatch')

@lru_cache(maxsize=128)
@torch.inference_mode()
def encode_query(query):
    batch = tokenizer(['query: ' + query], padding=True, truncation=True,
                      max_length=256, return_tensors='pt').to(DEVICE)
    hidden = e5_model(**batch).last_hidden_state
    mask = batch['attention_mask'].unsqueeze(-1)
    vector = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
    return F.normalize(vector.float(), p=2, dim=1).cpu().numpy().astype('float32')

STOP = frozenset('a an the for with to of and in on by from at best good new'.split())
GENERIC_CATEGORY = frozenset('electronics accessory accessories supply supplies other'.split())

def canonical_tokens(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[‐‑‒–—−]', '-', text)
    text = re.sub(r'\b(?:usb|type)[\s-]*c\b', ' usbc ', text)
    result = []
    for token in re.findall(r'\w+', text, flags=re.UNICODE):
        if token == 'mice':
            token = 'mouse'
        elif len(token) > 4 and token.endswith('ies'):
            token = token[:-3] + 'y'
        elif len(token) > 4 and token.endswith('s') and not token.endswith(('ss', 'us', 'is')):
            token = token[:-1]
        result.append(token)
    return tuple(result)

def content_tokens(text):
    return tuple(t for t in canonical_tokens(text) if t not in STOP)

def primary_title(text):
    # General role/compatibility boundaries; no query or model exceptions.
    return re.split(r'\b(?:for|with|compatible\s+with|designed\s+for|works\s+with)\b|[|;]',
                    str(text), maxsplit=1, flags=re.I)[0]

def has_phrase(tokens, phrase):
    return any(tokens[i:i + len(phrase)] == phrase
               for i in range(len(tokens) - len(phrase) + 1))

def prepare_metadata(catalog):
    full = [content_tokens(t) for t in catalog.title.fillna('')]
    head = [content_tokens(primary_title(t)) for t in catalog.title.fillna('')]
    leaf = [content_tokens(clean_field(s) or clean_field(p).split(' > ')[-1])
            for s, p in zip(catalog.subcategory, catalog.cat_path)]
    vocabulary = set()
    for tokens in leaf + [content_tokens(g) for g in catalog.product_group]:
        for part in range(1, min(3, len(tokens)) + 1):
            phrase = tokens[-part:]
            if phrase and not set(phrase) & GENERIC_CATEGORY:
                vocabulary.add(phrase)
    document_counts = Counter(t for tokens in full for t in set(tokens))
    idf = {t: math.log1p(len(catalog) / (count + 1)) for t, count in document_counts.items()}
    # Ephemeral metadata lookup only; never build/save a replacement search index.
    postings = defaultdict(set)
    for row, tokens in enumerate(head):
        for token in set(tokens) | set(leaf[row]):
            postings[token].add(row)
    return full, head, leaf, vocabulary, idf, postings


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

## 3 · Reusable search service

`SearchEngine` accepts loaded artifacts and encoder callables. It returns JSON-compatible dictionaries, stable `catalog_row` + `parent_asin`, explicit status, scores, and rejection diagnostics. It has no Drive mounting, display, file-writing, or HTTP fetching in request methods. Instantiate once at a future FastAPI startup; serialize model access as appropriate for the serving runtime.

Baseline reproduces notebook 04's selected **BM25 + E5, 100 candidates each, RRF k=60**. Improved retrieval adds bounded metadata candidates, then a bounded linear rerank; it never hard-filters by a guessed product type. Gaming/wireless/connector requirements remain ordinary query attributes. Scores are evidence, not probabilities. Image ranking remains CLIP-only, with no text rejection threshold.


In [5]:
# Components: normalized RRF, primary-title coverage, full-title coverage,
# category/role alignment, primary-title adjacent-phrase coverage.
# Fixed small grid, chosen BEFORE evaluating this revision.
RANKERS = {
    'baseline': None,
    'balanced': (0.50, 0.25, 0.10, 0.10, 0.05),
    'role_aware': (0.35, 0.30, 0.10, 0.20, 0.05),
}

class SearchEngine:
    def __init__(self, catalog, bm25, e5_index, e5_vectors, image_index, image_rows,
                 catalog_hash, encode_text, encode_image, candidate_k=100, rrf_k=60):
        self.catalog = catalog
        self.bm25, self.e5_index = bm25, e5_index
        self.e5_vectors, self.catalog_hash = e5_vectors, catalog_hash
        self.image_index, self.image_rows = image_index, image_rows
        self.encode_text, self.encode_image = encode_text, encode_image
        self.candidate_k, self.rrf_k = candidate_k, rrf_k
        self.unknown_idf = math.log1p(len(catalog))
        (self.full, self.head, self.leaf, self.types,
         self.idf, self.postings) = prepare_metadata(catalog)

    @staticmethod
    def validate_k(top_k):
        if isinstance(top_k, bool) or not isinstance(top_k, int) or not 1 <= top_k <= 100:
            raise ValueError('top_k must be an integer from 1 to 100')

    def intent(self, query):
        tokens = content_tokens(query)
        matches = [p for p in self.types if has_phrase(tokens, p)]
        # Prefer specific category phrases; stable ties, no evaluation labels used.
        role = min(matches, key=lambda p: (-len(p), -sum(self.idf.get(t, self.unknown_idf) for t in p), p)) if matches else ()
        return tokens, role

    def coverage(self, query_tokens, tokens):
        terms = set(query_tokens)
        total = sum(self.idf.get(t, self.unknown_idf) for t in terms)
        return sum(self.idf.get(t, self.unknown_idf) for t in terms & set(tokens)) / total if total else 0.0

    def features(self, row, tokens, role):
        primary = self.coverage(tokens, self.head[row])
        full = self.coverage(tokens, self.full[row])
        if not role:
            aligned = 0.0
        elif role[-1] in self.leaf[row]:
            aligned = self.coverage(role, self.head[row] + self.leaf[row])
        elif not self.leaf[row]:
            aligned = 0.5 * float(has_phrase(self.head[row], role))
        else:
            aligned = 0.0
        pairs = list(zip(tokens, tokens[1:]))
        phrase = sum(has_phrase(self.head[row], p) for p in pairs) / len(pairs) if pairs else primary
        return primary, full, aligned, phrase

    @lru_cache(maxsize=128)
    def retrieve(self, query, legacy=False):
        # Legacy mode is diagnostic: old notebook 05 changed BM25 tokenization and k.
        count = min(150 if legacy else self.candidate_k, len(self.catalog))
        vector = self.encode_text(query)
        scores, ids = self.e5_index.search(vector, count)
        dense = [int(i) for i in ids[0] if i >= 0]
        if legacy:
            text = query.lower().replace('usb-c', 'usb c').replace('type-c', 'usb c')
            words = re.findall(r'\w+', re.sub(r'[^\w\s]+', ' ', text))
        else:
            words = bm25_tokens(query)
        bm_scores = self.bm25.get_scores(words)
        lexical = np.argsort(-bm_scores, kind='stable')[:count].tolist()
        fused = defaultdict(float)
        for ranking in (dense, lexical):
            for rank, row in enumerate(ranking, 1):
                fused[row] += 1.0 / (self.rrf_k + rank)
        return vector, dict(fused), dense, lexical

    @lru_cache(maxsize=128)
    def rank_text(self, query, ranker='baseline', legacy=False):
        if ranker not in RANKERS:
            raise ValueError(f'Unknown ranker: {ranker}')
        vector, fused, dense, lexical = self.retrieve(query, legacy=legacy)
        tokens, role = self.intent(query)
        rows = set(fused)
        metadata_rows = set()
        if ranker != 'baseline':
            for token in set(tokens):
                metadata_rows.update(self.postings.get(token, ()))
            # Lexical rescue can find primary products missed by the original union.
            rescue = sorted(metadata_rows, key=lambda r: (
                -(self.features(r, tokens, role)[0] + self.features(r, tokens, role)[2]), r))
            rows.update(rescue[:self.candidate_k])
        max_rrf = max(fused.values(), default=1.0)
        records = []
        for row in sorted(rows):
            primary, full, aligned, phrase = self.features(row, tokens, role)
            parts = (fused.get(row, 0.0) / max_rrf, primary, full, aligned, phrase)
            score = fused.get(row, 0.0) if ranker == 'baseline' else sum(
                weight * value for weight, value in zip(RANKERS[ranker], parts))
            records.append({'catalog_row': int(row), 'ranking_score': float(score),
                            'rrf_score': float(fused.get(row, 0.0)),
                            'primary_coverage': float(primary), 'title_coverage': float(full),
                            'role_alignment': float(aligned), 'phrase_coverage': float(phrase),
                            'bm25_candidate': row in lexical, 'e5_candidate': row in dense,
                            'metadata_candidate': row in metadata_rows})
        records.sort(key=lambda item: (-item['ranking_score'], item['catalog_row']))
        return records, vector, role

    def serialize_product(self, record):
        row = self.catalog.iloc[record['catalog_row']]
        item = dict(record)
        for field in ('parent_asin', 'title', 'image_url', 'product_group', 'subcategory'):
            item[field] = clean_field(row.get(field), 4000)
        price = row.get('price')
        item['price'] = float(price) if price is not None and pd.notna(price) and np.isfinite(price) else None
        return item

    def text_evidence(self, records, vector, role):
        # Same product must supply semantic AND role/lexical evidence.
        # Scores for rescue/BM25-only candidates are exact dot products with saved E5 vectors.
        confidence = 0.0
        for record in records[:20]:
            row = record['catalog_row']
            semantic = float(np.dot(self.e5_vectors[row], vector[0]))
            lexical = record['primary_coverage']
            if role:
                lexical = min(lexical, record['role_alignment'])
            confidence = max(confidence, 0.75 * semantic + 0.25 * lexical)
        return float(confidence)

    def search_text(self, query, top_k=10, ranker='baseline', rejection=None, legacy=False):
        self.validate_k(top_k)
        if not isinstance(query, str):
            raise TypeError('query must be a string')
        query = query.strip()
        if not bm25_tokens(query):
            return {'status': 'empty_query', 'results': [], 'diagnostics': {'rejection_enabled': False}}
        if len(query) > 2000:
            raise ValueError('query exceeds 2000 characters')
        records, vector, role = self.rank_text(query, ranker, legacy=legacy)
        confidence = self.text_evidence(records, vector, role)
        enabled = bool(rejection and rejection.get('enabled'))
        if enabled:
            require(rejection.get('ranker') == ranker and not legacy
                    and rejection.get('catalog_hash') == self.catalog_hash
                    and rejection.get('validated') is True
                    and np.isfinite(rejection['threshold']), 'Rejection policy does not match this engine')
        reject = enabled and confidence < rejection['threshold']
        return {'status': 'no_match' if reject else 'ok',
                'results': [] if reject else [self.serialize_product(r) for r in records[:top_k]],
                'diagnostics': {'ranker': ranker, 'rejection_enabled': enabled,
                                'confidence': confidence, 'detected_role': list(role),
                                'candidate_count': len(records)}}

    def search_image(self, image, top_k=10):
        self.validate_k(top_k)
        if not isinstance(image, Image.Image):
            raise TypeError('image must be a decoded PIL.Image.Image')
        query = self.encode_image(ImageOps.exif_transpose(image).convert('RGB'))
        scores, ids = self.image_index.search(query, min(top_k, self.image_index.ntotal))
        records = [{'catalog_row': int(self.image_rows[int(i)]), 'similarity': float(score)}
                   for score, i in zip(scores[0], ids[0]) if i >= 0]
        return {'status': 'ok', 'results': [self.serialize_product(r) for r in records],
                'diagnostics': {'modality': 'image', 'rejection_enabled': False}}


In [6]:
# Lazy CLIP loading; this encodes a single query image, never the product catalog.
clip_model, clip_processor = None, None

@torch.inference_mode()
def encode_image_query(image):
    global clip_model, clip_processor
    if clip_model is None:
        clip_model = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
        clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
        require(clip_model.config.projection_dim == image_index.d, 'CLIP query dimension mismatch')
    batch = clip_processor(images=[image], return_tensors='pt').to(DEVICE)
    vector = clip_model.get_image_features(**batch)
    return F.normalize(vector.float(), p=2, dim=1).cpu().numpy().astype('float32')

engine = SearchEngine(products, bm25, e5_index, e5_vectors, image_index, image_rows,
                      catalog_hash, encode_query, encode_image_query, CANDIDATE_K, RRF_K)


In [16]:
pending = pd.read_csv(
    SAVE / "final_v2/pending_judgments_filled.csv"
)
labels = pd.read_csv(SAVE / "relevance_labels.csv")

merged = pd.concat([labels, pending], ignore_index=True)
merged = merged.drop_duplicates(
    subset=["query_id", "parent_asin"],
    keep="last"
)
merged.to_csv(SAVE / "relevance_labels.csv", index=False)

print("Total labels:", len(merged))

Total labels: 226


## 4 · Existing queries, judgments, and evaluation protocol

Read `search_improvement/evaluation_queries.csv` and `relevance_labels.csv` from the previous run; never overwrite them. Precision treats grades 1–2 as relevant; nDCG uses gains `2^grade−1`. Missing judgments are **unknown**, not zero. A paired comparison is withheld if any returned top-10 item is unjudged. IDCG uses the same existing judged pool for every system; it is not exhaustive catalog relevance. Short result lists retain denominator 10. All-zero pools give nDCG 0, matching the old convention; also inspect in-domain and ambiguous groups separately from out-of-domain.

**Leakage limitation:** all 16 queries and prior results were already inspected. The split below is retrospective validation, not an untouched final test. Known failure queries q03–q05 and related headphone queries stay in development. Choose weights on development only, then use validation once to accept or fall back to the original baseline. Do not repeatedly retune using validation results. A future independent judged set is required for a generalization claim.


In [17]:
queries_path = SAVE / 'evaluation_queries.csv'
labels_path = SAVE / 'relevance_labels.csv'
if not queries_path.is_file() or not labels_path.is_file():
    raise FileNotFoundError('Mount the existing search_improvement queries and human labels from Drive')
query_df = pd.read_csv(queries_path, dtype=str, keep_default_na=False)
judged = pd.read_csv(labels_path, dtype={'query_id': str, 'parent_asin': str})
require({'query_id', 'query', 'query_group'} <= set(query_df), 'Unexpected query schema')
require({'query_id', 'parent_asin', 'relevance'} <= set(judged), 'Unexpected label schema')
require(query_df.query_id.is_unique and query_df['query'].str.strip().ne('').all(),
        'Duplicate IDs or empty queries')
require(not judged.duplicated(['query_id', 'parent_asin']).any(), 'Duplicate relevance judgments')
require(set(judged.query_id) <= set(query_df.query_id)
        and set(judged.parent_asin) <= set(asins), 'Labels reference unknown queries/products')
numeric = pd.to_numeric(judged.relevance, errors='coerce')
require((judged.relevance.isna() | numeric.isin([0, 1, 2])).all(), 'Grades must be 0, 1, 2 or blank')
judged['relevance'] = numeric
for field in ('query', 'query_group'):
    if field in judged:
        expected = judged.query_id.map(query_df.set_index('query_id')[field])
        require(judged[field].eq(expected).all(), f'Label {field} differs from saved queries')
rel = {(r.query_id, r.parent_asin): int(r.relevance)
       for r in judged.itertuples() if pd.notna(r.relevance)}

# Predeclared split for the EXISTING 16 IDs, not a source of relevance labels.
DEV_IDS = {'q01', 'q02', 'q03', 'q04', 'q05', 'q09', 'q11', 'q13', 'q14'}
VALIDATION_IDS = {'q06', 'q07', 'q08', 'q10', 'q12', 'q15', 'q16'}
require(set(query_df.query_id) == DEV_IDS | VALIDATION_IDS
        and not DEV_IDS & VALIDATION_IDS, 'Existing query set changed; review the split before tuning')
require(set(query_df.query_group) <= {'in_domain', 'ambiguous', 'out_of_domain'},
        'Unexpected query groups')
query_df['split'] = np.where(query_df.query_id.isin(DEV_IDS), 'development', 'validation')
display(query_df)
print('Judged pairs:', len(rel), '| blank pairs:', int(judged.relevance.isna().sum()))


,query_id,query,query_group,split
0,q01,wireless noise cancelling headphones,in_domain,development
1,q02,Sony Bluetooth headphones,in_domain,development
2,q03,Logitech wireless gaming mouse,in_domain,development
3,q04,portable power bank for charging phone,in_domain,development
4,q05,USB-C fast charging cable,in_domain,development
5,q06,laptop stand,in_domain,validation
6,q07,4k computer monitor,in_domain,validation
7,q08,smartwatch for running,in_domain,validation
8,q09,wireless earbuds,in_domain,development
9,q10,gaming keyboard,in_domain,validation


Judged pairs: 226 | blank pairs: 0


## 5 · Ranking diagnostics and judgment coverage

Compare each component, candidate inclusion, category path, primary-title coverage and role alignment. The three historical failures below are diagnostics only; query text never controls ranking. Catalog counts distinguish a retrieval miss from possible catalog scarcity, but do not prove relevance or absence. New top-10 products may lie outside the old judgment pool: the final save cell writes a blank `pending_judgments.csv` for human review, preserving all existing grades. Add genuine judgments to the existing label file and rerun evaluation; do not infer grades from these features.


In [18]:
runs = {}
for query in query_df.itertuples(index=False):
    for name in RANKERS:
        runs[(query.query_id, name)] = engine.search_text(query.query, K, ranker=name)
    # Reproduces previous 05 baseline for continuity with the historical numbers.
    runs[(query.query_id, 'legacy05_baseline')] = engine.search_text(
        query.query, K, ranker='baseline', legacy=True)

for qid in ('q03', 'q04', 'q05'):
    query = query_df.set_index('query_id').loc[qid, 'query']
    tokens, role = engine.intent(query)
    matching = [i for i in range(len(products)) if engine.features(i, tokens, role)[2] >= 0.99] if role else []
    print('\n', qid, query, '| catalog role:', role or 'not recognized',
          '| strongly aligned catalog rows:', len(matching))
    for name in ('baseline', 'role_aware'):
        frame = pd.DataFrame(runs[(qid, name)]['results'])
        print(name)
        display(frame[['catalog_row', 'parent_asin', 'title', 'subcategory', 'ranking_score',
                       'primary_coverage', 'role_alignment', 'bm25_candidate', 'e5_candidate']].head(5))
    if matching:
        display(products.iloc[matching[:5]][['parent_asin', 'title', 'cat_path']])

coverage_rows, missing_pairs = [], {}
for query in query_df.itertuples(index=False):
    for name in [*RANKERS, 'legacy05_baseline']:
        results = runs[(query.query_id, name)]['results']
        unknown = [r for r in results if (query.query_id, r['parent_asin']) not in rel]
        coverage_rows.append({'query_id': query.query_id, 'split': query.split, 'system': name,
                              'returned': len(results), 'judged': len(results) - len(unknown),
                              'unjudged': len(unknown)})
        for item in unknown:
            missing_pairs[(query.query_id, item['parent_asin'])] = {
                'query_id': query.query_id, 'query': query.query, 'query_group': query.query_group,
                'parent_asin': item['parent_asin'], 'catalog_row': item['catalog_row'],
                'title': item['title'], 'relevance': None}
coverage_df = pd.DataFrame(coverage_rows)
pending_df = pd.DataFrame(list(missing_pairs.values()), columns=[
    'query_id', 'query', 'query_group', 'parent_asin', 'catalog_row', 'title', 'relevance'])
display(coverage_df.groupby(['split', 'system'])[['returned', 'judged', 'unjudged']].sum())
print('Unknown pairs needing human judgment:', len(pending_df))



 q03 Logitech wireless gaming mouse | catalog role: ('mouse',) | strongly aligned catalog rows: 1053
baseline


,catalog_row,parent_asin,title,subcategory,ranking_score,primary_coverage,role_alignment,bm25_candidate,e5_candidate
0,10368,B07K6ZVJCP,Logitech M310 red Full Size Wireless Mouse,Mice,0.028372,0.739932,1.0,True,True
1,10983,B01FZP1UY4,"Logitech M325 Mouse, Wireless Cocktail, (Cockt...",Mice,0.027402,0.739932,1.0,True,True
2,10301,B002Y6CJJ8,Logitech M305 Wireless Mouse (Silver),Mice,0.026999,0.739932,1.0,True,True
3,10001,B00552K0GM,Logitech 910-002235 Wireless Mouse M185,Mice,0.026736,0.739932,1.0,True,True
4,10868,B08436BCLX,Logitech Precision Pro Wireless Mouse,Mice,0.026334,0.739932,1.0,True,True


role_aware


,catalog_row,parent_asin,title,subcategory,ranking_score,primary_coverage,role_alignment,bm25_candidate,e5_candidate
0,10368,B07K6ZVJCP,Logitech M310 red Full Size Wireless Mouse,Mice,0.845973,0.739932,1.0,True,True
1,10983,B01FZP1UY4,"Logitech M325 Mouse, Wireless Cocktail, (Cockt...",Mice,0.834016,0.739932,1.0,True,True
2,10301,B002Y6CJJ8,Logitech M305 Wireless Mouse (Silver),Mice,0.829035,0.739932,1.0,True,True
3,10352,B0055QZ216,Logitech G Optical Gaming Mouse G400 with High...,Mice,0.827374,0.831035,1.0,True,True
4,10658,B004ZEZB7O,Logitech Wireless Mouse M305 (Victorian Wallpa...,Mice,0.827037,0.739932,1.0,True,True


,parent_asin,title,cat_path
9998,B0B4SWZTZ1,"seenda Wireless Mouse, 2.4G Noiseless Mouse wi...",Electronics > Computers & Accessories > Comput...
9999,B07538T649,Amazon Basics 3-Button Wired USB Computer Mous...,Electronics > Computers & Accessories > Comput...
10000,B082M94GY9,"Uiosmuph LED Wireless Mouse, G12 Slim Recharge...",Electronics > Computers & Accessories > Comput...
10001,B00552K0GM,Logitech 910-002235 Wireless Mouse M185,Electronics > Computers & Accessories > Comput...
10002,B071Y2SG3R,Amazon Basics Wireless Computer Mouse with USB...,Electronics > Computers & Accessories > Comput...



 q04 portable power bank for charging phone | catalog role: not recognized | strongly aligned catalog rows: 0
baseline


... (6 outputs left out)

,catalog_row,parent_asin,title,subcategory,ranking_score,primary_coverage,role_alignment,bm25_candidate,e5_candidate
0,12467,B07KPFDG8Q,Suptig Charging Cable Type-C Charging Cable 2 ...,Cables & Cords,0.824687,0.702231,1.0,True,True
1,12226,B07479LGLT,UGREEN Electronic Organizer Travel Cable Organ...,Cable Organizer Bags & Cases,0.687758,0.251418,1.0,True,True
2,12469,B07WGGSD23,Suptig Charging Cable Mini USB Charging Cable ...,Cables & Cords,0.674076,0.505978,1.0,True,True
3,12464,B07PQKC2M8,2 Pack 16.4FT Power Extension Cable Compatible...,Cables & Cords,0.622915,0.251418,1.0,True,True
4,12463,B07Z5MGFBL,"2 Pack 10FT Power Extension Cable for WyzeCam,...",Cables & Cords,0.605737,0.251418,1.0,True,True


,parent_asin,title,cat_path
12223,B07N1PF5JV,iMangoo Shockproof Carrying Case Hard Protecti...,Electronics > Accessories & Supplies > Cable O...
12224,B0C6CNNBG9,"FYY Electronic Organizer, Travel Cable Organiz...",Electronics > Accessories & Supplies > Cable O...
12225,B0B19TK33F,"MATEIN Cable Organizer Bag, Large Travel Tech ...",Electronics > Accessories & Supplies > Cable O...
12226,B07479LGLT,UGREEN Electronic Organizer Travel Cable Organ...,Electronics > Accessories & Supplies > Cable O...
12227,B0BD7L7RV4,"Luxtude Electronics Organizer, Portable Cable ...",Electronics > Accessories & Supplies > Cable O...


returned  judged  unjudged
split       system                                       
development balanced                 90      90         0
            baseline                 90      90         0
            legacy05_baseline        90      90         0
            role_aware               90      90         0
validation  balanced                 70      70         0
            baseline                 70      70         0
            legacy05_baseline        70      70         0
            role_aware               70      70         0

Unknown pairs needing human judgment: 0


In [19]:
METRICS = ['precision@10', 'mrr@10', 'ndcg@10']

def ranking_metrics(qid, results):
    keys = [(qid, r['parent_asin']) for r in results[:K]]
    if any(key not in rel for key in keys):
        return None
    gains = [rel[key] for key in keys]
    binary = [int(g > 0) for g in gains]
    ideal = sorted([g for (q, _), g in rel.items() if q == qid], reverse=True)[:K]
    dcg = sum((2**g - 1) / math.log2(rank + 2) for rank, g in enumerate(gains))
    idcg = sum((2**g - 1) / math.log2(rank + 2) for rank, g in enumerate(ideal))
    return {'precision@10': sum(binary) / K,
            'mrr@10': next((1 / rank for rank, b in enumerate(binary, 1) if b), 0.0),
            'ndcg@10': dcg / idcg if idcg else 0.0}

def paired_evaluation(ids, systems, run_map):
    rows = []
    for query in query_df[query_df.query_id.isin(ids)].itertuples(index=False):
        # Do not let incomplete labels silently change the query subset.
        for name in systems:
            metrics = ranking_metrics(query.query_id, run_map[(query.query_id, name)]['results'])
            if metrics is None:
                return pd.DataFrame()
            rows.append({'query_id': query.query_id, 'query_group': query.query_group,
                         'split': query.split, 'system': name, **metrics})
    return pd.DataFrame(rows)

def non_regressing(candidate, baseline, require_improvement=True):
    delta = candidate[METRICS] - baseline[METRICS]
    return bool((delta >= -1e-12).all() and (not require_improvement or (delta > 1e-12).any()))

dev_metrics = paired_evaluation(DEV_IDS, list(RANKERS), runs)
chosen_ranker = 'baseline'
selection_reason = 'Development judgments incomplete; retain notebook 04 baseline'
if not dev_metrics.empty:
    dev_means = dev_metrics.groupby('system')[METRICS].mean()
    display(dev_means)
    eligible = [name for name in RANKERS if name != 'baseline'
                and non_regressing(dev_means.loc[name], dev_means.loc['baseline'])]
    if eligible:
        chosen_ranker = max(eligible, key=lambda name: (
            *dev_means.loc[name, ['ndcg@10', 'mrr@10', 'precision@10']].tolist(), name))
        selection_reason = 'Chosen on development only; all three means non-regressing'
    else:
        selection_reason = 'No development candidate improves without regression; retain baseline'
print('Frozen development choice:', chosen_ranker, '|', selection_reason)


,precision@10,mrr@10,ndcg@10
system,,,
balanced,0.633333,0.777778,0.681003
baseline,0.622222,0.703704,0.621089
role_aware,0.633333,0.777778,0.685047


Frozen development choice: role_aware | Chosen on development only; all three means non-regressing


## 6 · One validation comparison and final ranker

The validation set is used only for this frozen choice. Report paired per-query deltas and means; never substitute a different heuristic after seeing validation. Deployment falls back to baseline if judgments are incomplete or any of the three mean ranking metrics regresses. Aggregate improvements do not guarantee improvement for every query. The full 16-query table is descriptive and includes development data, not final test performance.


In [20]:
validation_systems = list(dict.fromkeys(['baseline', chosen_ranker]))
validation_metrics = paired_evaluation(VALIDATION_IDS, validation_systems, runs)
final_ranker = 'baseline'
validation_reason = 'Incomplete validation judgments; baseline retained'
if not validation_metrics.empty:
    means = validation_metrics.groupby('system')[METRICS].mean()
    display(means)
    if chosen_ranker != 'baseline' and non_regressing(means.loc[chosen_ranker], means.loc['baseline']):
        final_ranker = chosen_ranker
        validation_reason = 'Frozen choice passes all three aggregate validation checks'
    elif chosen_ranker == 'baseline':
        validation_reason = 'Development selected baseline'
    else:
        validation_reason = 'Validation did not confirm non-regressing improvement; baseline retained'
    if chosen_ranker != 'baseline':
        left = validation_metrics.query("system == 'baseline'").set_index('query_id')[METRICS]
        right = validation_metrics[validation_metrics.system == chosen_ranker].set_index('query_id')[METRICS]
        display((right - left).add_prefix('delta_'))
print('Final ranker:', final_ranker, '|', validation_reason)

all_metrics = paired_evaluation(set(query_df.query_id), validation_systems, runs)
if not all_metrics.empty:
    print('DESCRIPTIVE, reused full set (not an independent final test):')
    display(all_metrics.groupby('system')[METRICS].mean())
    display(all_metrics.groupby(['query_group', 'system'])[METRICS].mean())
else:
    print('Full-set ranking metrics withheld: missing judgments; see coverage report.')

historical_check = paired_evaluation(set(query_df.query_id), ['legacy05_baseline'], runs)
if not historical_check.empty:
    print('Historical 05 baseline reproduction (k=150, legacy query tokenization):')
    display(historical_check.groupby('system')[METRICS].mean())
print('The primary baseline is notebook 04 (k=100, original BM25 tokenizer); old 05 numbers may differ.')


,precision@10,mrr@10,ndcg@10
system,,,
baseline,0.542857,0.571429,0.473021
role_aware,0.571429,0.571429,0.556950


,delta_precision@10,delta_mrr@10,delta_ndcg@10
query_id,,,
q06,0.0,0.0,0.000000
q07,0.0,0.0,0.000000
q08,0.0,0.0,0.323241
q10,0.1,0.0,0.110046
q12,0.1,0.0,0.154215
q15,0.0,0.0,0.000000
q16,0.0,0.0,0.000000


Final ranker: role_aware | Frozen choice passes all three aggregate validation checks
DESCRIPTIVE, reused full set (not an independent final test):


,precision@10,mrr@10,ndcg@10
system,,,
baseline,0.58750,0.645833,0.556309
role_aware,0.60625,0.687500,0.629005


precision@10    mrr@10   ndcg@10
query_group   system                                      
ambiguous     baseline            0.95  1.000000  0.922892
              role_aware          1.00  1.000000  1.000000
in_domain     baseline            0.75  0.833333  0.705516
              role_aware          0.77  0.900000  0.806407
out_of_domain baseline            0.00  0.000000  0.000000
              role_aware          0.00  0.000000  0.000000

Historical 05 baseline reproduction (k=150, legacy query tokenization):


,precision@10,mrr@10,ndcg@10
system,,,
legacy05_baseline,0.5875,0.645833,0.556309


The primary baseline is notebook 04 (k=100, original BM25 tokenizer); old 05 numbers may differ.


## 7 · No-match calibration and validation (text only)

Use the existing out-of-domain query groups as negative query evidence and existing positive relevance grades as support evidence. **An all-zero judged pool does not establish that the catalog has no matching product.** Such queries remain `unknown_support`, never silently become unsupported labels.

To evaluate unsupported intents, a human may add `catalog_support` to the **existing** `evaluation_queries.csv`, with `supported` or `unsupported` only after checking the catalog; leave uncertain cases blank. No new queries or automatic labels are created. These optional declarations are query-level evidence, separate from product relevance grades; contradictions fail validation. Without confirmed unsupported examples in both splits, calibration is reported but rejection remains disabled. This is an explicit remaining data limitation of the original labels.

Confidence combines absolute E5 cosine and primary/role evidence from the **same product** among the top 20. Develop a scalar threshold on development only, minimizing false acceptance subject to zero false rejection. Validate the frozen threshold once. Enable only with supported, out-of-domain **and confirmed unsupported** cases in both splits and zero observed false rejection/acceptance in validation. This strict check on a tiny reused set is a provisional safeguard, not a statistical guarantee. No text threshold applies to image search.


In [21]:
support_rows = []
for query in query_df.itertuples(index=False):
    positive = any(qid == query.query_id and grade > 0 for (qid, _), grade in rel.items())
    declaration = str(getattr(query, 'catalog_support', '')).strip().lower()
    require(declaration in {'', 'supported', 'unsupported'}, 'Invalid catalog_support value')
    if query.query_group == 'out_of_domain':
        require(not positive and declaration != 'supported', 'Out-of-domain evidence contradicts labels')
        target = 'out_of_domain'
    elif declaration == 'unsupported':
        require(not positive, 'Unsupported declaration contradicts positive relevance judgments')
        target = 'unsupported'
    elif positive or declaration == 'supported':
        target = 'supported'
    else:
        target = 'unknown_support'
    # Always recalculated for the exact final ranker, before applying any rejection.
    response = engine.search_text(query.query, K, ranker=final_ranker)
    support_rows.append({'query_id': query.query_id, 'query': query.query, 'split': query.split,
                         'query_group': query.query_group, 'support_evidence': target,
                         'confidence': response['diagnostics']['confidence']})
calibration_df = pd.DataFrame(support_rows)
display(calibration_df)

def gate_counts(frame, threshold):
    known = frame[frame.support_evidence != 'unknown_support']
    positives = known.support_evidence.eq('supported')
    rejected = known.confidence.lt(threshold)
    supported_n = int(positives.sum())
    negative_n = int((~positives).sum())
    false_reject = int((positives & rejected).sum())
    false_accept = int((~positives & ~rejected).sum())
    result = {'supported_n': supported_n, 'negative_n': negative_n,
              'unknown_n': int((frame.support_evidence == 'unknown_support').sum()),
              'false_reject': false_reject, 'false_accept': false_accept,
              'false_rejection_rate': false_reject / supported_n if supported_n else None,
              'false_acceptance_rate': false_accept / negative_n if negative_n else None}
    for label in ('out_of_domain', 'unsupported'):
        mask = known.support_evidence.eq(label)
        result[label + '_n'] = int(mask.sum())
        result[label + '_false_accept'] = int((mask & ~rejected).sum())
    return result

dev_cal = calibration_df.query("split == 'development'")
val_cal = calibration_df.query("split == 'validation'")
values = sorted(dev_cal.loc[dev_cal.support_evidence != 'unknown_support', 'confidence'].unique())
thresholds = [0.0] + [(a + b) / 2 for a, b in zip(values, values[1:])]
if values:
    thresholds.append(float(np.nextafter(values[-1], np.inf)))
curve_df = pd.DataFrame([{'threshold': float(t), **gate_counts(dev_cal, t)} for t in thresholds])
eligible = curve_df[(curve_df.false_reject == 0) & (curve_df.supported_n > 0)
                    & (curve_df.negative_n > 0)]
threshold = None
gate_policy = {'enabled': False, 'validated': False, 'ranker': final_ranker,
               'catalog_hash': catalog_hash, 'threshold': None,
               'reason': 'Insufficient development evidence'}
gate_validation = {}
if not eligible.empty:
    best = eligible.sort_values(['false_accept', 'threshold'], kind='stable').iloc[0]
    threshold = float(best['threshold'])
    gate_validation = gate_counts(val_cal, threshold)
    dev_complete = all(int(best[name + '_n']) > 0 for name in ('out_of_domain', 'unsupported'))
    val_complete = all(gate_validation[name + '_n'] > 0 for name in ('out_of_domain', 'unsupported'))
    enabled = bool(dev_complete and val_complete and int(best.false_accept) == 0
                   and gate_validation['supported_n'] > 0
                   and gate_validation['false_reject'] == 0 and gate_validation['false_accept'] == 0)
    gate_policy.update(enabled=enabled, validated=enabled, threshold=threshold,
                       reason=('Passed frozen development/validation checks' if enabled else
                               'Insufficient class coverage or failed frozen validation; rejection disabled'))
display(curve_df)
print('Frozen threshold validation:', gate_validation)
print('Text rejection policy:', gate_policy)

# Show effect on ranking separately; never silently replace the ungated comparison.
gated_runs = dict(runs)
for query in query_df.itertuples(index=False):
    gated_runs[(query.query_id, 'final_with_gate')] = engine.search_text(
        query.query, K, ranker=final_ranker, rejection=gate_policy)
gated_metrics = paired_evaluation(VALIDATION_IDS, ['baseline', 'final_with_gate'], gated_runs)
if not gated_metrics.empty:
    print('Validation ranking after the evaluated policy (identical ungated results if disabled):')
    display(gated_metrics.groupby('system')[METRICS].mean())


,query_id,query,split,query_group,support_evidence,confidence
0,q01,wireless noise cancelling headphones,development,in_domain,supported,0.940690
1,q02,Sony Bluetooth headphones,development,in_domain,supported,0.929898
2,q03,Logitech wireless gaming mouse,development,in_domain,supported,0.910560
3,q04,portable power bank for charging phone,development,in_domain,supported,0.782301
4,q05,USB-C fast charging cable,development,in_domain,supported,0.816995
5,q06,laptop stand,validation,in_domain,unknown_support,0.792678
6,q07,4k computer monitor,validation,in_domain,supported,0.914612
7,q08,smartwatch for running,validation,in_domain,supported,0.672408
8,q09,wireless earbuds,development,in_domain,supported,0.929710
9,q10,gaming keyboard,validation,in_domain,supported,0.919333


,threshold,supported_n,negative_n,unknown_n,false_reject,false_accept,false_rejection_rate,false_acceptance_rate,out_of_domain_n,out_of_domain_false_accept,unsupported_n,unsupported_false_accept
0,0.000000,7,2,0,0,2,0.000000,1.0,2,2,0,0
1,0.717512,7,2,0,0,1,0.000000,0.5,2,1,0,0
2,0.764373,7,2,0,0,0,0.000000,0.0,2,0,0,0
3,0.799648,7,2,0,1,0,0.142857,0.0,2,0,0,0
4,0.853733,7,2,0,2,0,0.285714,0.0,2,0,0,0
5,0.900515,7,2,0,3,0,0.428571,0.0,2,0,0,0
6,0.920135,7,2,0,4,0,0.571429,0.0,2,0,0,0
7,0.929804,7,2,0,5,0,0.714286,0.0,2,0,0,0
8,0.935294,7,2,0,6,0,0.857143,0.0,2,0,0,0
9,0.940690,7,2,0,7,0,1.000000,0.0,2,0,0,0


Frozen threshold validation: {'supported_n': 4, 'negative_n': 2, 'unknown_n': 1, 'false_reject': 1, 'false_accept': 0, 'false_rejection_rate': 0.25, 'false_acceptance_rate': 0.0, 'out_of_domain_n': 2, 'out_of_domain_false_accept': 0, 'unsupported_n': 0, 'unsupported_false_accept': 0}
Text rejection policy: {'enabled': False, 'validated': False, 'ranker': 'role_aware', 'catalog_hash': '4f1e28ecb05ad3eb', 'threshold': 0.7643730760009011, 'reason': 'Insufficient class coverage or failed frozen validation; rejection disabled'}
Validation ranking after the evaluated policy (identical ungated results if disabled):


,precision@10,mrr@10,ndcg@10
system,,,
baseline,0.542857,0.571429,0.473021
final_with_gate,0.571429,0.571429,0.556950


## 8 · Separate image-search verification

CLIP ranking is unchanged. First check saved-vector/index/mapping consistency; this is **not image relevance evaluation**. Then upload one or a few local query images to Colab to test preprocessing → CLIP → saved FAISS → product IDs. No catalog images or large datasets are downloaded. This smoke check must run before claiming image input verification.

The existing relevance labels are for text queries, not images. Do not assign image Precision/MRR/nDCG from them. Neither image quality improvement nor image no-match calibration can be claimed without image judgments. Repeated catalog images can tie at cosine 1, so verify vector equivalence rather than requiring a particular tied self-ID in top-k.


In [12]:
image_checks = []
for pos in np.linspace(0, len(image_rows) - 1, min(16, len(image_rows)), dtype=int):
    row = int(image_rows[pos])
    vector = np.ascontiguousarray(image_vectors[row:row + 1], dtype='float32')
    scores, ids = image_index.search(vector, min(10, image_index.ntotal))
    mapped = image_rows[ids[0][ids[0] >= 0]]
    require(len(mapped) > 0 and np.isclose(scores[0, 0], 1.0, atol=1e-3), 'Image self similarity failed')
    require(any(np.allclose(image_vectors[int(r)], vector[0], atol=1e-5) for r in mapped),
            'No equivalent image vector returned')
    require(image_valid[mapped].all(), 'Returned invalid image row')
    image_checks.append({'catalog_row': row, 'parent_asin': asins[row],
                         'equivalent_top10': True, 'top_similarity': float(scores[0, 0])})
image_mapping_df = pd.DataFrame(image_checks)
display(image_mapping_df)
print('Saved CLIP index/mapping verified. Raw image input still requires the next cell.')


,catalog_row,parent_asin,equivalent_top10,top_similarity
0,0,B08XPWDSWW,True,1.0
1,888,B09QPKX8RF,True,1.0
2,1776,B0882DBX66,True,1.0
3,2665,B0BB2J41ZR,True,1.0
4,3553,B07GC4SRKN,True,1.0
5,4442,B0B2K6JSBR,True,1.0
6,5330,B0765ZFGLZ,True,1.0
7,6219,B0CH83GNJM,True,1.0
8,7107,B07N8WRV89,True,1.0
9,7996,B09CLJJV7X,True,1.0


Saved CLIP index/mapping verified. Raw image input still requires the next cell.


In [13]:
# Interactive Colab smoke test. Cancel/skip if no query images are available;
# final metadata explicitly records that input verification is incomplete.
from google.colab import files
from io import BytesIO

uploaded = files.upload()
image_input_checks = []
for filename, payload in uploaded.items():
    try:
        with Image.open(BytesIO(payload)) as image:
            response = engine.search_image(image, top_k=5)
        json.dumps(response, allow_nan=False)
        require(len(response['results']) > 0, 'No image results')
        for item in response['results']:
            row = item['catalog_row']
            require(image_valid[row] and item['parent_asin'] == asins[row], 'Image product mapping mismatch')
        print(filename, '| Image pipeline completed; visually inspect relevance:')
        display(pd.DataFrame(response['results'])[['catalog_row', 'parent_asin', 'title', 'similarity']])
        image_input_checks.append({'file': filename, 'pipeline_passed': True})
    except (OSError, ValueError, TypeError) as exc:
        print(filename, '| Failed:', exc)
        image_input_checks.append({'file': filename, 'pipeline_passed': False, 'error': str(exc)})
if not image_input_checks:
    print('Raw-image input verification NOT performed; saved-vector mapping checks only.')


Saving R1_76d63d9f-1147-4b03-8d83-b2799a44a9b2.webp to R1_76d63d9f-1147-4b03-8d83-b2799a44a9b2.webp


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

R1_76d63d9f-1147-4b03-8d83-b2799a44a9b2.webp | Image pipeline completed; visually inspect relevance:


,catalog_row,parent_asin,title,similarity
0,10356,B0BC2STR39,"Verbatim Wireless Notebook Optical Mouse, Comm...",0.910519
1,10385,B00IJ2E05C,Microsoft Sculpt Mobile Mouse - Light Orchid (...,0.891967
2,10727,B0BF5NHPPX,"RAPIQUE Bluetooth Wireless Mouse, Rechargeable...",0.887680
3,10080,B07X7TKDV2,"seenda Wireless Mouse, 2.4G Wireless Computer ...",0.885737
4,10751,B0BKGL8YYJ,"seenda Silent Wireless Mouse, Cute Soft 2.4G C...",0.879613


## 9 · Text contract checks and final artifacts

These Colab checks validate reusable request/response contracts and product mappings, not relevance labels. Save only a versioned configuration/evaluation report and (if needed) blank pending judgment rows. The existing query/label files, catalog, embeddings, BM25 and FAISS artifacts are never overwritten or duplicated. All saves stay under Drive's `search_improvement/final_v2/`; source code remains in this notebook.


In [22]:
require(engine.search_text('   ')['status'] == 'empty_query', 'Empty-query contract failed')
require(engine.search_text('---')['results'] == [], 'Punctuation-query contract failed')
require(canonical_tokens('USB-C') == canonical_tokens('type c') == canonical_tokens('USB–C'),
        'Connector normalization failed')
require(bm25_tokens('USB-C') == ['usb', 'c'], 'Saved BM25 tokenizer changed')
for query in query_df.itertuples(index=False):
    first = engine.search_text(query.query, K, ranker=final_ranker, rejection=gate_policy)
    second = engine.search_text(query.query, K, ranker=final_ranker, rejection=gate_policy)
    require(first == second, 'Non-deterministic ranking')
    json.dumps(first, allow_nan=False)
    rows = [r['catalog_row'] for r in first['results']]
    require(len(rows) == len(set(rows)), 'Duplicate result rows')
    for item in first['results']:
        require(item['parent_asin'] == asins[item['catalog_row']], 'Text product mapping mismatch')
print('Text API contracts, deterministic ordering, JSON serialization and ID mappings passed.')

# Thin entry points to reuse after loading this notebook's service at app startup.
def search_text(query, top_k=10):
    return engine.search_text(query, top_k, ranker=final_ranker, rejection=gate_policy)

def search_image(image, top_k=10):
    return engine.search_image(image, top_k)


Text API contracts, deterministic ordering, JSON serialization and ID mappings passed.


In [23]:
FINAL = SAVE / 'final_v2'
FINAL.mkdir(parents=True, exist_ok=True)

def records(frame):
    # Convert pandas/numpy scalars and NaN to strict JSON values.
    return json.loads(frame.to_json(orient='records'))

config = {
    'schema_version': 2, 'catalog_hash': catalog_hash, 'n_products': len(products),
    'models': {'e5': E5_ID, 'clip': CLIP_ID},
    'asset_directory_relative_to_project_root': 'search_assets',
    'asset_files': required, 'e5_row_mapping': 'FAISS position = catalog_row',
    'image_row_mapping': 'clip_image_row_ids.npy[FAISS position] = catalog_row',
    'image_row_mapping_sha256': hashlib.sha256(image_rows.astype('<i8').tobytes()).hexdigest(),
    'product_id': 'parent_asin', 'candidate_k': CANDIDATE_K, 'rrf_k': RRF_K,
    'ranker_candidates': RANKERS, 'development_choice': chosen_ranker,
    'final_ranker': final_ranker, 'selection_reason': selection_reason,
    'validation_reason': validation_reason, 'text_rejection': gate_policy,
    'image_ranking': 'unchanged CLIP inner product', 'image_rejection_enabled': False,
    'development_ids': sorted(DEV_IDS), 'validation_ids': sorted(VALIDATION_IDS),
    'evaluation_k': K, 'unjudged_policy': 'withhold incomplete paired comparison',
    'label_sha256': hashlib.sha256(labels_path.read_bytes()).hexdigest(),
    'query_sha256': hashlib.sha256(queries_path.read_bytes()).hexdigest(),
    'evaluation_limitation': 'Retrospective reused-query validation; pooled IDCG; no independent final test',
    'packages': {name: version(name) for name in
                 ['numpy', 'pandas', 'torch', 'transformers', 'faiss-cpu', 'rank-bm25']},
    'python': platform.python_version(),
}
report = {
    'development_ranking': records(dev_metrics), 'validation_ranking': records(validation_metrics),
    'descriptive_full_set_ranking': records(all_metrics),
    'ranking_outputs': [
        {'query_id': query.query_id, 'system': name, 'rank': rank,
         'judged_grade': rel.get((query.query_id, item['parent_asin'])), **item}
        for query in query_df.itertuples(index=False)
        for name in validation_systems
        for rank, item in enumerate(runs[(query.query_id, name)]['results'], 1)],
    'legacy05_baseline_reproduction': records(historical_check),
    'coverage': records(coverage_df), 'text_rejection_evidence': records(calibration_df),
    'development_threshold_curve': records(curve_df), 'gate_validation': gate_validation,
    'gated_validation_ranking': records(gated_metrics),
    'image_mapping_checks': records(image_mapping_df),
    'image_input_checks': globals().get('image_input_checks', []),
    'image_relevance_evaluated': False,
}
for filename, value in [('search_config.json', config), ('evaluation_report.json', report)]:
    (FINAL / filename).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
if not pending_df.empty:
    pending_df.to_csv(FINAL / 'pending_judgments.csv', index=False)
elif (FINAL / 'pending_judgments.csv').exists():
    # Remove only our stale generated blank-review file, never the source labels.
    (FINAL / 'pending_judgments.csv').unlink()
print('Saved:', FINAL)
print('Final ranker:', final_ranker, '| text no-match enabled:', gate_policy['enabled'])
print('Unknown relevance pairs:', len(pending_df), '| image inputs checked:',
      len(globals().get('image_input_checks', [])))


Saved: /content/drive/MyDrive/ElectroLens/search_improvement/final_v2
Final ranker: role_aware | text no-match enabled: False
Unknown relevance pairs: 0 | image inputs checked: 1


## Run and interpret

1. Open this notebook in Colab, select a T4 if available, mount the correct Drive, and run in order. Do not rerun notebooks 03–04 to rebuild assets.
2. Inspect diagnostics and judgment coverage. If comparisons are withheld, review `final_v2/pending_judgments.csv` and merge genuine grades into the existing `relevance_labels.csv`, preserving prior judgments. Rerun from evaluation. Do not repeatedly change ranking weights in response to validation.
3. Review no-match counts by class. Keep rejection disabled unless the frozen policy passes; confirm catalog support for existing unsupported queries if possible. All-zero result pools alone are insufficient.
4. Upload small query images for the separate CLIP input smoke check, inspect returned products, then run contract checks and save final reports. A skipped upload is recorded as unverified.

**Remaining limits:** actual revised metrics require the Drive artifacts and human judgments; static editing cannot establish improvement. The small reused query set cannot support a clean held-out claim. Category/title heuristics depend on metadata quality, connector/model synonyms are incomplete, and the English lexical layer may underperform for multilingual queries. A candidate cap can still miss relevant products. Catalog absence requires catalog-level review; threshold calibration needs confirmed unsupported cases. Image relevance and image rejection remain unevaluated. Production extraction should load the same validated row mappings, models, chosen configuration, and service once, then add request size/concurrency controls in the future backend.
